## Notebook de exploración API Open Charge Map

In [1]:
# Libreria para cargar variables de entorno
import os 
from dotenv import load_dotenv

#Libreria para conectar a api
import requests

# Libreria json
import json

# Libreria pandas
import pandas as pd

# Cargar variables de entorno
load_dotenv()
api_key = os.getenv("API_KEY")
# print(api_key)

### Datos referenciales

In [13]:
# Obtener datos de referencia 
url_ref = "https://api.openchargemap.io/v3/referencedata"
params_ref = {
    "output": "json",          
    "key": api_key         
}

response_ref = requests.get(url_ref, params = params_ref)
data_ref = response_ref.json()
# print(data_ref)

#### Paises

In [14]:
# Obtener paises
paises = data_ref["Countries"]

# Ver estructura
print(json.dumps(paises[0], indent=4)) 

{
    "ISOCode": "GB",
    "ContinentCode": "EU",
    "ID": 1,
    "Title": "United Kingdom"
}


In [15]:
# Convertir en data frame
df_paises = pd.json_normalize(paises)

# Revisar tipos de datos 
print("\nTipos de datos----------------")
print(df_paises.dtypes)


Tipos de datos----------------
ISOCode            str
ContinentCode      str
ID               int64
Title              str
dtype: object


In [17]:
# Revisar cantidad total de registros
print(len(df_paises))

250


In [16]:
# Revisar primeros elementos
df_paises.head()

,ISOCode,ContinentCode,ID,Title
0,GB,EU,1,United Kingdom
1,US,NA,2,United States
2,IE,EU,3,Ireland
3,HK,AS,4,Hong Kong
4,AF,AS,5,Afghanistan


In [19]:
# Revisar cantidad max de caracteres de continent code e iso code
print(df_paises['ContinentCode'].str.len().max())
print(df_paises['ISOCode'].str.len().max())

2.0
2


In [21]:
# Buscar Chile
print(df_paises.query('Title == "Chile"'))

   ISOCode ContinentCode  ID  Title
48      CL            SA  49  Chile


#### Tipos de conexiones

**ConnectionType** Tipo de conexión para usuario final con soporte EVSE

In [48]:
# Obtener tipos de conexiones
tipos_conexiones = data_ref["ConnectionTypes"]

# Revisar estructura
print(json.dumps(tipos_conexiones[0], indent=4))

{
    "FormalName": "Avcon SAE J1772-2001",
    "IsDiscontinued": true,
    "IsObsolete": false,
    "ID": 7,
    "Title": "Avcon Connector"
}


In [49]:
# Convertir en data frame
df_tipos_conexiones = pd.json_normalize(tipos_conexiones)

# Consultar los primeros elementos 
df_tipos_conexiones.head()

,FormalName,IsDiscontinued,IsObsolete,ID,Title
0,Avcon SAE J1772-2001,True,False,7,Avcon Connector
1,NaN,None,None,4,Blue Commando (2P+E)
2,BS1363 / Type G,None,None,3,BS1363 3 Pin 13 Amp
3,IEC 62196-3 Configuration EE,False,False,32,CCS (Type 1)
4,IEC 62196-3 Configuration FF,False,False,33,CCS (Type 2)


In [50]:
# Revisar tipos de datos
print("\nTipos de datos------------")
print(df_tipos_conexiones.dtypes)

# Consultar totalidad de filas
print("\nTotal de filas------------")
print(len(df_tipos_conexiones))


Tipos de datos------------
FormalName           str
IsDiscontinued    object
IsObsolete        object
ID                 int64
Title                str
dtype: object

Total de filas------------
43


In [52]:
# Revisar valores nulos
df_tipos_conexiones.isnull().sum()

FormalName        16
IsDiscontinued     5
IsObsolete         5
ID                 0
Title              0
dtype: int64

#### Operadores

**Operators** Operadores, organización que controla la red de puntos de carga.

In [5]:
# Consultar operadores
operadores = data_ref["Operators"]

# Revisar estructura
print(json.dumps(operadores[0], indent=4))

{
    "WebsiteURL": null,
    "Comments": "For use when the operator of the equipment is a single business owner connected to the location and equipment is not part of a larger network",
    "PhonePrimaryContact": null,
    "PhoneSecondaryContact": null,
    "IsPrivateIndividual": false,
    "AddressInfo": null,
    "BookingURL": null,
    "ContactEmail": null,
    "FaultReportEmail": null,
    "IsRestrictedEdit": null,
    "ID": 45,
    "Title": "(Business Owner at Location)"
}


In [6]:
# Convertir a data frame
df_operadores = pd.json_normalize(operadores)

# Revisar estructura 
print("\nEstructura de operadores----------")
print(df_operadores.dtypes)


Estructura de operadores----------
WebsiteURL                  str
Comments                    str
PhonePrimaryContact         str
PhoneSecondaryContact       str
IsPrivateIndividual      object
AddressInfo              object
BookingURL                  str
ContactEmail                str
FaultReportEmail            str
IsRestrictedEdit         object
ID                        int64
Title                       str
dtype: object


In [7]:
# Revisar valores nulos
print(df_operadores.isnull().sum())

WebsiteURL                13
Comments                 845
PhonePrimaryContact      596
PhoneSecondaryContact    941
IsPrivateIndividual       21
AddressInfo              966
BookingURL               965
ContactEmail             687
FaultReportEmail         926
IsRestrictedEdit          49
ID                         0
Title                      0
dtype: int64


In [8]:
# Obtener dataframe solo con las columnas necesarias
df_operadores_sub = pd.DataFrame({
    "ID": df_operadores["ID"],
    "Title": df_operadores["Title"],
    "IsPrivateIndividual": df_operadores["IsPrivateIndividual"],
    "WebsiteURL": df_operadores["WebsiteURL"]
})

# Ver primeros elementos
df_operadores_sub.head()


,ID,Title,IsPrivateIndividual,WebsiteURL
0,45,(Business Owner at Location),False,NaN
1,44,(Private Residence/Individual),True,NaN
2,1,(Unknown Operator),None,NaN
3,3871,5 Şarj (TR),False,https://5sarj.com
4,3697,7-Charge (7-Eleven),False,https://www.7-eleven.com/7charge


#### Tipos de Estados

**StatusType** Estado de una locación o item de equipamiento que indica si la estación esta generalmente operativa

In [56]:
# Obtener estados
estados = data_ref["StatusTypes"]

# Ver estructura
print(json.dumps(estados[0], indent=2))

# Convertir en data frame
df_estados = pd.json_normalize(estados)

{
  "IsOperational": null,
  "IsUserSelectable": true,
  "ID": 0,
  "Title": "Unknown"
}


In [57]:
# Revisar tipos de datos
print("\nTipos de datos------------")
print(df_estados.dtypes)


Tipos de datos------------
IsOperational       object
IsUserSelectable      bool
ID                   int64
Title                  str
dtype: object


In [58]:
# Revisar valores nulos
print(df_estados.isnull().sum())

IsOperational       1
IsUserSelectable    0
ID                  0
Title               0
dtype: int64


In [59]:
# Ver primeros registros
df_estados.head()

,IsOperational,IsUserSelectable,ID,Title
0,None,True,0,Unknown
1,True,False,10,Currently Available (Automated Status)
2,True,False,20,Currently In Use (Automated Status)
3,True,True,30,Temporarily Unavailable
4,True,True,50,Operational


#### Tipo de suministro

**SupplyType** Indica el tipo de poder de suministro EVSE

In [9]:
# Obtener estados
suministro = data_ref["CurrentTypes"]

# Revisar estructura
print(json.dumps(suministro[0], indent=2))

# Convertir en data frame
df_suministro = pd.json_normalize(suministro)

{
  "Description": "Alternating Current - Single Phase",
  "ID": 10,
  "Title": "AC (Single-Phase)"
}


In [10]:
# Revisar tipos de datos
print(df_suministro.dtypes)

Description      str
ID             int64
Title            str
dtype: object


In [11]:
# Revisar valores nulos
print(df_suministro.isnull().sum())

Description    0
ID             0
Title          0
dtype: int64


In [12]:
# Ver primeros registros
df_suministro.head()

,Description,ID,Title
0,Alternating Current - Single Phase,10,AC (Single-Phase)
1,Alternating Current - Three Phase,20,AC (Three-Phase)
2,Direct Current,30,DC


### POI

In [89]:
# Puntos de carga 
url_poi = "https://api.openchargemap.io/v3/poi"

# Parametros busqueda por id de pais Chile
params_country = {
    "output": "json",      
    "countryid" : [49],
    "maxresults": 500,         
    "key": api_key           
}

response_poi = requests.get(url_poi, params = params_country)
data_poi = response_poi.json()

In [90]:
# Estructura completa
print(json.dumps(data_poi[0], indent=4))

{
    "DataProvider": {
        "WebsiteURL": "http://openchargemap.org",
        "Comments": null,
        "DataProviderStatusType": {
            "IsProviderEnabled": true,
            "ID": 1,
            "Title": "Manual Data Entry"
        },
        "IsRestrictedEdit": false,
        "IsOpenDataLicensed": true,
        "IsApprovedImport": true,
        "License": "Licensed under Creative Commons Attribution 4.0 International (CC BY 4.0)",
        "DateLastImported": null,
        "ID": 1,
        "Title": "Open Charge Map Contributors"
    },
    "OperatorInfo": {
        "WebsiteURL": null,
        "Comments": null,
        "PhonePrimaryContact": null,
        "PhoneSecondaryContact": null,
        "IsPrivateIndividual": null,
        "AddressInfo": null,
        "BookingURL": null,
        "ContactEmail": null,
        "FaultReportEmail": null,
        "IsRestrictedEdit": null,
        "ID": 1,
        "Title": "(Unknown Operator)"
    },
    "UsageType": {
        "IsPayAtLoca

In [91]:
# Covertir en dataframe con los valores que necesitamos 
df_pois = pd.json_normalize(data_poi)

# Consultar total de resultados, será total de pois en chile
print(len(df_pois))

184


#### POIs

In [92]:
# Definir dataframe solo con las columnas que interesan
df_pois_sel = df_pois[[
    "ID",
    "AddressInfo.Title",
    "OperatorID",
    "AddressInfo.AddressLine1",
    "AddressInfo.AddressLine2",
    "AddressInfo.Town",
    "AddressInfo.StateOrProvince",
    "AddressInfo.CountryID",
    "AddressInfo.Latitude",
    "AddressInfo.Longitude"
]]

# Mostrar dataframe final 
df_pois_sel.head()

,ID,AddressInfo.Title,OperatorID,AddressInfo.AddressLine1,AddressInfo.AddressLine2,AddressInfo.Town,AddressInfo.StateOrProvince,AddressInfo.CountryID,AddressInfo.Latitude,AddressInfo.Longitude
0,474881,Cargador Gratuito Casablanca,1,Avenida Constitución 11,NaN,Casablanca,Casablanca,49,-33.320179,-71.410707
1,472300,BOULEVARD NOGALES MACHALI,80,Carretera Del Cobre 2521 km 4,Machali Rancagua,NaN,Región del Libertador General Bernardo O'Higgins,49,-34.191053,-70.705057
2,470298,Espacio Urbano Linares,80,Avenida Aníbal León Bustos,NaN,NaN,Región del Maule,49,-35.843847,-71.606172
3,470297,Copec Voltex,3364,Calle 2 Norte N°2310,NaN,NaN,Región del Maule,49,-35.427619,-71.642291
4,297232,Copec Voltex Charging Station,3364,Av. Pdte. Manuel Bulnes 4486,NaN,Punta Arenas,Magallanes and Chilean Antarctica,49,-53.126978,-70.872706


In [93]:
# Revisar valores nulos
print(df_pois_sel.isnull().sum())

ID                               0
AddressInfo.Title                0
OperatorID                       0
AddressInfo.AddressLine1         0
AddressInfo.AddressLine2       157
AddressInfo.Town                 4
AddressInfo.StateOrProvince     14
AddressInfo.CountryID            0
AddressInfo.Latitude             0
AddressInfo.Longitude            0
dtype: int64


In [94]:
# Unir las dos columnas de dirección
df_pois_sel["Address"] = (
    df_pois_sel["AddressInfo.AddressLine1"].fillna("") + " " + df_pois_sel["AddressInfo.AddressLine2"].fillna("")
).str.strip()

# Revisar 
df_pois_sel.head()

,ID,AddressInfo.Title,OperatorID,AddressInfo.AddressLine1,AddressInfo.AddressLine2,AddressInfo.Town,AddressInfo.StateOrProvince,AddressInfo.CountryID,AddressInfo.Latitude,AddressInfo.Longitude,Address
0,474881,Cargador Gratuito Casablanca,1,Avenida Constitución 11,NaN,Casablanca,Casablanca,49,-33.320179,-71.410707,Avenida Constitución 11
1,472300,BOULEVARD NOGALES MACHALI,80,Carretera Del Cobre 2521 km 4,Machali Rancagua,NaN,Región del Libertador General Bernardo O'Higgins,49,-34.191053,-70.705057,Carretera Del Cobre 2521 km 4 Machali Rancagua
2,470298,Espacio Urbano Linares,80,Avenida Aníbal León Bustos,NaN,NaN,Región del Maule,49,-35.843847,-71.606172,Avenida Aníbal León Bustos
3,470297,Copec Voltex,3364,Calle 2 Norte N°2310,NaN,NaN,Región del Maule,49,-35.427619,-71.642291,Calle 2 Norte N°2310
4,297232,Copec Voltex Charging Station,3364,Av. Pdte. Manuel Bulnes 4486,NaN,Punta Arenas,Magallanes and Chilean Antarctica,49,-53.126978,-70.872706,Av. Pdte. Manuel Bulnes 4486


In [95]:
# Eliminar las columnas concatenadas
df_pois_sel = df_pois_sel.drop(columns = ["AddressInfo.AddressLine1","AddressInfo.AddressLine2"])

# Revisar 
df_pois_sel.head()

,ID,AddressInfo.Title,OperatorID,AddressInfo.Town,AddressInfo.StateOrProvince,AddressInfo.CountryID,AddressInfo.Latitude,AddressInfo.Longitude,Address
0,474881,Cargador Gratuito Casablanca,1,Casablanca,Casablanca,49,-33.320179,-71.410707,Avenida Constitución 11
1,472300,BOULEVARD NOGALES MACHALI,80,NaN,Región del Libertador General Bernardo O'Higgins,49,-34.191053,-70.705057,Carretera Del Cobre 2521 km 4 Machali Rancagua
2,470298,Espacio Urbano Linares,80,NaN,Región del Maule,49,-35.843847,-71.606172,Avenida Aníbal León Bustos
3,470297,Copec Voltex,3364,NaN,Región del Maule,49,-35.427619,-71.642291,Calle 2 Norte N°2310
4,297232,Copec Voltex Charging Station,3364,Punta Arenas,Magallanes and Chilean Antarctica,49,-53.126978,-70.872706,Av. Pdte. Manuel Bulnes 4486


In [96]:
# Cambiar nombre de columnas
df_pois_sel = df_pois_sel.rename(columns = {
    "ID": "POI_Id",
    "AddressInfo.Title": "Name",
    "OperatorID": "Operator_Id",
    "AddressInfo.Town": "Town",
    "AddressInfo.StateOrProvince": "StateOrProvince",
    "AddressInfo.CountryID": "Country_Id",
    "AddressInfo.Latitude": "Latitude",
    "AddressInfo.Longitude": "Longitude",
    "Address": "Address"
})

# Revisar
df_pois_sel.head()

,POI_Id,Name,Operator_Id,Town,StateOrProvince,Country_Id,Latitude,Longitude,Address
0,474881,Cargador Gratuito Casablanca,1,Casablanca,Casablanca,49,-33.320179,-71.410707,Avenida Constitución 11
1,472300,BOULEVARD NOGALES MACHALI,80,NaN,Región del Libertador General Bernardo O'Higgins,49,-34.191053,-70.705057,Carretera Del Cobre 2521 km 4 Machali Rancagua
2,470298,Espacio Urbano Linares,80,NaN,Región del Maule,49,-35.843847,-71.606172,Avenida Aníbal León Bustos
3,470297,Copec Voltex,3364,NaN,Región del Maule,49,-35.427619,-71.642291,Calle 2 Norte N°2310
4,297232,Copec Voltex Charging Station,3364,Punta Arenas,Magallanes and Chilean Antarctica,49,-53.126978,-70.872706,Av. Pdte. Manuel Bulnes 4486


In [97]:
# Reordenar columnas
df_pois_sel = df_pois_sel[["POI_Id","Name","Operator_Id","Country_Id","StateOrProvince","Town","Address","Latitude","Longitude"]]

# Revisar 
df_pois_sel.head()

,POI_Id,Name,Operator_Id,Country_Id,StateOrProvince,Town,Address,Latitude,Longitude
0,474881,Cargador Gratuito Casablanca,1,49,Casablanca,Casablanca,Avenida Constitución 11,-33.320179,-71.410707
1,472300,BOULEVARD NOGALES MACHALI,80,49,Región del Libertador General Bernardo O'Higgins,NaN,Carretera Del Cobre 2521 km 4 Machali Rancagua,-34.191053,-70.705057
2,470298,Espacio Urbano Linares,80,49,Región del Maule,NaN,Avenida Aníbal León Bustos,-35.843847,-71.606172
3,470297,Copec Voltex,3364,49,Región del Maule,NaN,Calle 2 Norte N°2310,-35.427619,-71.642291
4,297232,Copec Voltex Charging Station,3364,49,Magallanes and Chilean Antarctica,Punta Arenas,Av. Pdte. Manuel Bulnes 4486,-53.126978,-70.872706


In [98]:
# Revision de nulos
df_pois_sel.isnull().sum()

POI_Id              0
Name                0
Operator_Id         0
Country_Id          0
StateOrProvince    14
Town                4
Address             0
Latitude            0
Longitude           0
dtype: int64

#### Conexiones (Por cada POI)

In [104]:
# Revisar estrusctura de conexiones
print(json.dumps(data_poi[0]["Connections"][0],indent=4))

{
    "ID": 797205,
    "ConnectionTypeID": 25,
    "ConnectionType": {
        "FormalName": "IEC 62196-2 Type 2",
        "IsDiscontinued": false,
        "IsObsolete": false,
        "ID": 25,
        "Title": "Type 2 (Socket Only)"
    },
    "Reference": null,
    "StatusTypeID": 50,
    "StatusType": {
        "IsOperational": true,
        "IsUserSelectable": true,
        "ID": 50,
        "Title": "Operational"
    },
    "LevelID": null,
    "Level": null,
    "Amps": null,
    "Voltage": null,
    "PowerKW": null,
    "CurrentTypeID": 20,
    "CurrentType": {
        "Description": "Alternating Current - Three Phase",
        "ID": 20,
        "Title": "AC (Three-Phase)"
    },
    "Quantity": 2,
    "Comments": null
}


In [110]:
# Obtener las conexiones, array de objetos dentro de poi
# Se expanden las conexiones y se mantiene el ID como referencia. Se incorpora prefijo para evitar choque de nombres 
df_conexiones = pd.json_normalize(
    data_poi,
    record_path = ["Connections"],
    meta = ["ID"],
    meta_prefix = "Poi_"
)

# Revisar primeros elementos 
df_conexiones.head()

,ID,ConnectionTypeID,Reference,StatusTypeID,LevelID,Level,Amps,Voltage,PowerKW,CurrentTypeID,...,StatusType.Title,CurrentType.Description,CurrentType.ID,CurrentType.Title,Level.Comments,Level.IsFastChargeCapable,Level.ID,Level.Title,CurrentType,Poi_ID
0,797205,25,NaN,50,NaN,NaN,NaN,NaN,NaN,20.0,...,Operational,Alternating Current - Three Phase,20.0,AC (Three-Phase),NaN,NaN,NaN,NaN,NaN,474881
1,793120,33,NaN,50,3.0,NaN,NaN,NaN,60.0,30.0,...,Operational,Direct Current,30.0,DC,40KW and Higher,True,3.0,Level 3: High (Over 40kW),NaN,472300
2,793121,33,NaN,50,3.0,NaN,NaN,NaN,60.0,30.0,...,Operational,Direct Current,30.0,DC,40KW and Higher,True,3.0,Level 3: High (Over 40kW),NaN,472300
3,789611,33,NaN,50,3.0,NaN,NaN,NaN,60.0,30.0,...,Operational,Direct Current,30.0,DC,40KW and Higher,True,3.0,Level 3: High (Over 40kW),NaN,470298
4,789612,33,NaN,50,3.0,NaN,NaN,NaN,60.0,30.0,...,Operational,Direct Current,30.0,DC,40KW and Higher,True,3.0,Level 3: High (Over 40kW),NaN,470298


In [111]:
# Consultar cantidad de resultados
print(len(df_conexiones))

311


In [112]:
# Revisar tipos de datos
print(df_conexiones.dtypes)

ID                                 int64
ConnectionTypeID                   int64
Reference                            str
StatusTypeID                       int64
LevelID                          float64
Level                            float64
Amps                             float64
Voltage                          float64
PowerKW                          float64
CurrentTypeID                    float64
Quantity                         float64
Comments                          object
ConnectionType.FormalName            str
ConnectionType.IsDiscontinued     object
ConnectionType.IsObsolete         object
ConnectionType.ID                  int64
ConnectionType.Title                 str
StatusType.IsOperational            bool
StatusType.IsUserSelectable         bool
StatusType.ID                      int64
StatusType.Title                     str
CurrentType.Description              str
CurrentType.ID                   float64
CurrentType.Title                    str
Level.Comments  

In [114]:
# Seleccionar solo las columnas necesarias
df_conexiones_sel = df_conexiones[[
    "Poi_ID",
    "ID",
    "ConnectionTypeID",
    "StatusTypeID",
    "Amps",
    "Voltage",
    "PowerKW",
    "CurrentTypeID",
    "Quantity"
]]

# Revisar resultados
df_conexiones_sel.head()

,Poi_ID,ID,ConnectionTypeID,StatusTypeID,Amps,Voltage,PowerKW,CurrentTypeID,Quantity
0,474881,797205,25,50,NaN,NaN,NaN,20.0,2.0
1,472300,793120,33,50,NaN,NaN,60.0,30.0,1.0
2,472300,793121,33,50,NaN,NaN,60.0,30.0,1.0
3,470298,789611,33,50,NaN,NaN,60.0,30.0,1.0
4,470298,789612,33,50,NaN,NaN,60.0,30.0,1.0


In [115]:
# Revisar nulos
print(df_conexiones_sel.isnull().sum())

Poi_ID                0
ID                    0
ConnectionTypeID      0
StatusTypeID          0
Amps                280
Voltage             282
PowerKW              20
CurrentTypeID       219
Quantity              2
dtype: int64


In [117]:
# Renombrar columnas
df_conexiones_sel = df_conexiones_sel.rename(columns = {
    "Poi_ID": "Poi_Id",
    "ID": "Connection_Id",
    "ConnectionTypeID": "ConnectionType_Id",
    "StatusTypeID": "StatusType_Id",
    "CurrentTypeID": "CurrentType_Id"
})

# Revisar 
df_conexiones_sel.head()

,Poi_Id,Connection_Id,ConnectionType_Id,StatusType_Id,Amps,Voltage,PowerKW,CurrentType_Id,Quantity
0,474881,797205,25,50,NaN,NaN,NaN,20.0,2.0
1,472300,793120,33,50,NaN,NaN,60.0,30.0,1.0
2,472300,793121,33,50,NaN,NaN,60.0,30.0,1.0
3,470298,789611,33,50,NaN,NaN,60.0,30.0,1.0
4,470298,789612,33,50,NaN,NaN,60.0,30.0,1.0


In [118]:
# Ordenar columnas
df_conexiones_sel = df_conexiones_sel[[
    "Poi_Id",
    "Connection_Id",
    "ConnectionType_Id",
    "CurrentType_Id",
    "StatusType_Id",
    "Amps",
    "Voltage",
    "PowerKW",
    "Quantity"
]]

# Revisar 
df_conexiones_sel.head()

,Poi_Id,Connection_Id,ConnectionType_Id,CurrentType_Id,StatusType_Id,Amps,Voltage,PowerKW,Quantity
0,474881,797205,25,20.0,50,NaN,NaN,NaN,2.0
1,472300,793120,33,30.0,50,NaN,NaN,60.0,1.0
2,472300,793121,33,30.0,50,NaN,NaN,60.0,1.0
3,470298,789611,33,30.0,50,NaN,NaN,60.0,1.0
4,470298,789612,33,30.0,50,NaN,NaN,60.0,1.0
